In [ ]:
#!/usr/bin/env python3

import argparse
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


REQUIRED_CHECKPOINT_KEYS = (
    "depth_features",
    "depth_normal_features",
    "predictions",
    "targets",
)


def load_analysis_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        missing = [
            key
            for key in REQUIRED_CHECKPOINT_KEYS
            if key not in checkpoint
        ]

        if missing:
            raise KeyError(
                "Checkpoint is missing required arrays: "
                + ", ".join(missing)
            )

        arrays = {
            key: np.asarray(checkpoint[key])
            for key in REQUIRED_CHECKPOINT_KEYS
        }

    return arrays


def validate_arrays(
    depth_features,
    depth_normal_features,
    predictions,
    targets,
):
    depth_features = np.asarray(
        depth_features,
        dtype=np.float32,
    )

    depth_normal_features = np.asarray(
        depth_normal_features,
        dtype=np.float32,
    )

    predictions = np.asarray(
        predictions,
        dtype=np.float32,
    ).reshape(-1)

    targets = np.asarray(
        targets,
        dtype=np.float32,
    ).reshape(-1)

    if depth_features.ndim < 2:
        raise ValueError(
            "depth_features must contain one feature vector per sample"
        )

    if depth_normal_features.ndim < 2:
        raise ValueError(
            "depth_normal_features must contain one feature vector per sample"
        )

    n_samples = targets.shape[0]

    if n_samples < 2:
        raise ValueError(
            "At least two samples are required for latent-space analysis"
        )

    if predictions.shape[0] != n_samples:
        raise ValueError(
            "predictions and targets must have the same number of samples"
        )

    if depth_features.shape[0] != n_samples:
        raise ValueError(
            "depth_features and targets must have the same number of samples"
        )

    if depth_normal_features.shape[0] != n_samples:
        raise ValueError(
            "depth_normal_features and targets must have the same number of samples"
        )

    arrays = (
        depth_features,
        depth_normal_features,
        predictions,
        targets,
    )

    if not all(np.all(np.isfinite(array)) for array in arrays):
        raise ValueError(
            "All checkpoint arrays must contain only finite values"
        )

    return arrays


def prepare_features(
    features,
    random_state=42,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    if features.ndim > 2:
        features = features.reshape(
            features.shape[0],
            -1,
        )

    n_components = min(
        50,
        features.shape[1],
        features.shape[0] - 1,
    )

    if n_components < 2:
        return features

    pca = PCA(
        n_components=n_components,
        random_state=random_state,
    )

    return pca.fit_transform(features)


def run_tsne(
    features,
    perplexities=(10, 50),
    random_state=42,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    n_samples = features.shape[0]

    if n_samples < 2:
        raise ValueError(
            "At least two samples are required for t-SNE"
        )

    valid_perplexities = sorted(
        {
            float(perplexity)
            for perplexity in perplexities
            if 0 < float(perplexity) < n_samples
        }
    )

    if not valid_perplexities:
        valid_perplexities = [
            float(
                max(
                    1,
                    min(
                        30,
                        n_samples - 1,
                    ),
                )
            )
        ]

    tsne_args = {
        "n_components": 2,
        "metric": "euclidean",
        "random_state": random_state,
        "learning_rate": "auto",
        "init": "pca",
    }

    if "max_iter" in inspect.signature(TSNE).parameters:
        tsne_args["max_iter"] = 1000
    else:
        tsne_args["n_iter"] = 1000

    results = {}

    for perplexity in valid_perplexities:
        tsne = TSNE(
            perplexity=perplexity,
            **tsne_args,
        )

        embedding = tsne.fit_transform(
            features
        )

        results[perplexity] = {
            "embedding": embedding,
            "kl_divergence": float(
                getattr(
                    tsne,
                    "kl_divergence_",
                    np.nan,
                )
            ),
        }

    return results


def cluster_features(
    features,
    n_clusters=8,
    random_state=42,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    cluster_count = max(
        1,
        min(
            int(n_clusters),
            features.shape[0],
        ),
    )

    if cluster_count == 1:
        return np.zeros(
            features.shape[0],
            dtype=np.int32,
        )

    model = KMeans(
        n_clusters=cluster_count,
        random_state=random_state,
        n_init=20,
    )

    return model.fit_predict(
        features
    )


def normalize_values(values):
    values = np.asarray(
        values,
        dtype=np.float32,
    ).reshape(-1)

    minimum = float(
        np.min(values)
    )

    maximum = float(
        np.max(values)
    )

    if maximum > minimum:
        return (
            values - minimum
        ) / (
            maximum - minimum
        )

    return np.zeros_like(
        values,
        dtype=np.float32,
    )


def format_perplexity(perplexity):
    value = float(perplexity)

    if value.is_integer():
        return str(
            int(value)
        )

    return str(value).replace(
        ".",
        "p",
    )


def plot_tsne_error(
    embedding,
    errors,
    perplexity,
    output_path,
):
    embedding = np.asarray(
        embedding
    )

    normalized_errors = normalize_values(
        errors
    )

    figure, axis = plt.subplots(
        figsize=(10, 10)
    )

    scatter = axis.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=normalized_errors,
        cmap="viridis",
        s=60,
        alpha=0.5,
        edgecolors="gray",
        linewidths=0.3,
    )

    colorbar = figure.colorbar(
        scatter,
        ax=axis,
    )

    colorbar.set_label(
        "Normalized absolute error"
    )

    axis.set_title(
        f"t-SNE, perplexity = {perplexity:g}"
    )

    axis.set_xticks([])
    axis.set_yticks([])

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


def plot_tsne_clusters(
    embedding,
    labels,
    perplexity,
    output_path,
):
    embedding = np.asarray(
        embedding
    )

    labels = np.asarray(
        labels
    ).reshape(-1)

    figure, axis = plt.subplots(
        figsize=(10, 10)
    )

    scatter = axis.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=60,
        alpha=0.7,
        edgecolors="gray",
        linewidths=0.3,
    )

    colorbar = figure.colorbar(
        scatter,
        ax=axis,
    )

    colorbar.set_label(
        "K-means cluster"
    )

    axis.set_title(
        f"t-SNE clusters, perplexity = {perplexity:g}"
    )

    axis.set_xticks([])
    axis.set_yticks([])

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


def make_tsne_outputs(
    checkpoint_path,
    output_prefix="latent_space",
    perplexities=(10, 50),
    n_clusters=8,
    color_by="target",
    random_state=42,
):
    checkpoint = load_analysis_checkpoint(
        checkpoint_path
    )

    (
        depth_features,
        depth_normal_features,
        predictions,
        targets,
    ) = validate_arrays(
        checkpoint["depth_features"],
        checkpoint["depth_normal_features"],
        checkpoint["predictions"],
        checkpoint["targets"],
    )

    if color_by == "target":
        errors = np.abs(
            targets
        )
    elif color_by == "residual":
        errors = np.abs(
            predictions - targets
        )
    else:
        raise ValueError(
            "color_by must be 'target' or 'residual'"
        )

    feature_sets = {
        "depth": prepare_features(
            depth_features,
            random_state=random_state,
        ),
        "depth_normal": prepare_features(
            depth_normal_features,
            random_state=random_state,
        ),
    }

    output_prefix = Path(
        output_prefix
    )

    if output_prefix.parent != Path("."):
        output_prefix.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    outputs = {}
    save_data = {
        "errors": errors,
        "predictions": predictions,
        "targets": targets,
    }

    for feature_name, features in feature_sets.items():
        labels = cluster_features(
            features,
            n_clusters=n_clusters,
            random_state=random_state,
        )

        tsne_results = run_tsne(
            features,
            perplexities=perplexities,
            random_state=random_state,
        )

        outputs[feature_name] = {
            "clusters": labels,
            "perplexities": {},
        }

        save_data[
            f"{feature_name}_clusters"
        ] = labels

        for perplexity, result in tsne_results.items():
            perplexity_name = format_perplexity(
                perplexity
            )

            embedding = result[
                "embedding"
            ]

            error_plot_path = Path(
                f"{output_prefix}_{feature_name}_perp"
                f"{perplexity_name}_error.png"
            )

            cluster_plot_path = Path(
                f"{output_prefix}_{feature_name}_perp"
                f"{perplexity_name}_clusters.png"
            )

            plot_tsne_error(
                embedding,
                errors,
                perplexity,
                error_plot_path,
            )

            plot_tsne_clusters(
                embedding,
                labels,
                perplexity,
                cluster_plot_path,
            )

            outputs[feature_name][
                "perplexities"
            ][perplexity] = {
                "embedding": embedding,
                "kl_divergence": result[
                    "kl_divergence"
                ],
                "error_plot": str(
                    error_plot_path
                ),
                "cluster_plot": str(
                    cluster_plot_path
                ),
            }

            save_data[
                f"{feature_name}_perp"
                f"{perplexity_name}_embedding"
            ] = embedding

            save_data[
                f"{feature_name}_perp"
                f"{perplexity_name}_kl_divergence"
            ] = np.asarray(
                result["kl_divergence"],
                dtype=np.float64,
            )

    np.savez_compressed(
        f"{output_prefix}_data.npz",
        **save_data,
    )

    return outputs


def parse_perplexities(values):
    perplexities = tuple(
        float(value)
        for value in values.split(",")
        if value.strip()
    )

    if not perplexities:
        raise ValueError(
            "At least one perplexity must be provided"
        )

    return perplexities


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--checkpoint",
        required=True,
    )

    parser.add_argument(
        "--output-prefix",
        default="latent_space",
    )

    parser.add_argument(
        "--perplexities",
        default="10,50",
    )

    parser.add_argument(
        "--clusters",
        type=int,
        default=8,
    )

    parser.add_argument(
        "--color-by",
        choices=(
            "target",
            "residual",
        ),
        default="target",
    )

    parser.add_argument(
        "--random-state",
        type=int,
        default=42,
    )

    args = parser.parse_args()

    make_tsne_outputs(
        checkpoint_path=args.checkpoint,
        output_prefix=args.output_prefix,
        perplexities=parse_perplexities(
            args.perplexities
        ),
        n_clusters=args.clusters,
        color_by=args.color_by,
        random_state=args.random_state,
    )


if __name__ == "__main__":
    main()
